#RS Datacatlog Uploader

Data attributes DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537155636

Interaction events DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537352234

In [203]:
import dotenv
import math
from typing import List, Dict
import json
dotenv.load_dotenv()

import os
import requests
import pandas as pd
import re

RS_API_KEY = os.getenv("RUDDERSTACK_API_KEY")
RS_API_URL = "https://api.rudderstack.com"

headers = {
    "Authorization": f"Bearer {RS_API_KEY}",
    "Content-Type": "application/json",
}

data_attributes_filepath = "data/Data attributes.csv" # path to data attributes file exported from confluence
interaction_events_filepath = "data/Interaction events.csv" # path to interaction events file exported from confluence
required_events_filepath = "data/Required_properties.csv" # path to file holding list of required properties for events
tracking_plan_name = "MAC website tracking" # Set this for tracking plan name
category_name = "MAC website"  # Set this for category name put on created events

# global vars
category_id = "" # Leave blank
df_existing_properties = pd.DataFrame()
df_existing_events = pd.DataFrame()


In [207]:
# functions
def save_event_id(event_name:str, event_id:str) -> bool:
    """
    Update event in df_event_index with passed event id
    
    params:
      event_name: name of event to update
      param event_id: event_id
      
    returns:
      True if event updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if event_name in df_event_index['event'].values:
        df_event_index.loc[df_event_index['event'] == event_name, 'event_id'] = event_id
        rc = True
    
    return rc

def save_property_id(property_name:str, property_type:str, property_id:str) -> bool:
    """
    Update all events in df_event_index that have this property with the passed property id.\n 
    
    params:
      property_name: name of property to update
      property_type: type of property to update
      property_id: id of property\n

    returns:
      True if properties updated successfully, otherwise False
    """
    rc = False
    if not(df_event_index.empty):
      if not(df_event_index[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type)]).empty:
        df_event_index.loc[(df_event_index['property'] == property_name) & (df_event_index['type'] == property_type), 'property_id'] = property_id
        rc = True
    
    return rc

def get_property_id(name:str) -> str|None:
    """
    Find existing property ID using passed property name.
    
    params:
      property_id: id of property to find
      
    returns:
      property name if found, otherwise None
    """
    rc = None
    # df_existing_properties 
    if not(df_existing_properties.empty):
      if name in df_existing_properties['name'].values: 
        rc = df_existing_properties.loc[df_existing_properties['name'] == name, 'id'].values[0]
    
    return rc
  
def get_event_id(name:str) -> str|None:
    """
    Find existing event ID using passed event name.
    
    params:
      event_id: id of event to find
      
    returns:
      event name if found, otherwise None
    """
    rc = None
    
    # df_exsiting_events
    if not(df_existing_events.empty):
      if name in df_existing_events['name'].values:
        rc = df_existing_events.loc[df_existing_events['name'] == name, 'id'].values[0]
    
    return rc
  
def get_existing_properties():
    """
    Get existing properties from data catalog.  Update df_existing_properties dataframe with results.
    """
    global df_existing_properties
    df_existing_properties = pd.DataFrame() # clear dataframe
    
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/properties?page=1&orderBy=name:asc", headers=headers )
    #print(json.dumps(response.json(), indent=4, sort_keys=True))

    properties = []
    if response.status_code == 200:
        properties = response.json()['data']

        # Calculate total pages    
        total_rows = response.json()['total']
        page_size = response.json()['pageSize']
        total_pages = math.ceil(total_rows / page_size)

        # Retrieve any remaining pages
        if (total_pages > 1):
            for page in range(2, total_pages + 1):
                response = requests.get(f"{RS_API_URL}/v2/catalog/properties?page={page}&orderBy=name:asc", headers=headers )
                
                if response.status_code == 200:
                    properties.extend(response.json()['data'])
    else:
        print(f"Error: [{response.status_code}] {response.json()['error']}")

    df_existing_properties = pd.DataFrame(properties)
    if df_existing_properties.empty:
        print("No existing properties found")
    else:
        print(f"Found {len(df_existing_properties)} existing properties")
        
def get_existing_events():
    """
    Get existing events from data catalog.  Update df_existing_events dataframe with results.
    """
    global df_existing_events
    df_existing_events = pd.DataFrame() # clear dataframe
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/events?page=1&orderBy=name:asc", headers=headers )
    #print(json.dumps(response.json(), indent=4, sort_keys=True))

    events = []
    if response.status_code == 200:
        events = response.json()['data']

        # Calculate total pages    
        total_rows = response.json()['total']
        page_size = response.json()['pageSize']
        total_pages = math.ceil(total_rows / page_size)

        # Retrieve any remaining pages
        if (total_pages > 1):
            for page in range(2, total_pages + 1):
                response = requests.get(f"{RS_API_URL}/v2/catalog/events?page={page}&orderBy=name:asc", headers=headers )
                
                if response.status_code == 200:
                    events.extend(response.json()['data'])
    else:
        print(f"Error: [{response.status_code}] {response.json()['error']}")
        

    df_existing_events = pd.DataFrame(events)
    if df_existing_events.empty:
        print("No existing events found")
    else:
        print(f"Found {len(df_existing_events)} existing events")

In [208]:
# Retrieve existing properties and events from data catalog
get_existing_properties() 
get_existing_events()

No existing properties found
Found 10 existing events


In [160]:
# import interaction events file
df_events = pd.read_csv(interaction_events_filepath)
df_events['Trigger'] = df_events['Trigger'].fillna('blank') # default empty trigger values
df_events['Trigger'] = df_events['Trigger'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_events['Trigger'] = df_events['Trigger'].str.strip() # remove leading/trailing spaces

# import data attributes file
valid_types = ['object','string','number','integer','array','boolean','null']
df_properties = pd.read_csv(data_attributes_filepath)
df_properties = df_properties[df_properties['Data attribute'].notna()] # remove rows with missing 'Data attribute' values
df_properties['Type'] = df_properties['Type'].str.lower().str.strip() # lowercase all values and remove leading/trailing whitespace 
df_properties['Type'] = df_properties['Type'].replace(['object', 'array', 'list'], 'string') # change 'object', 'array', and 'list'and  types to 'string' 
df_properties['Description'] = df_properties['Description'].fillna('blank') # default empty description values
df_properties['Description'] = df_properties['Description'].apply(lambda x: re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', x)) # remove control characters (e.g. '\n')
df_properties['Description'] = df_properties['Description'].str.strip() # remove leading/trailing spaces 

# Check properties have valid types, otherwise remove property and log error
invalid_rows = df_properties[~df_properties['Type'].isin(valid_types)]
if len(invalid_rows) > 0:
    print(f"Found {len(invalid_rows)} rows with invalid types:")
    for idx, row in invalid_rows.iterrows():
        print(f"  Row {idx}: '{row['Data attribute']}' has invalid type '{row['Type']}'")
    df_properties = df_properties[df_properties['Type'].isin(valid_types)]


In [161]:
# Build an indexing list of properties within an event.  Set required to False as default.  

#  Process properties data, capture property type as property name/type is unique.
property_rows = []
for idx, property in df_properties.iterrows():
    event_list = property['Associated with'].split('\n')
    for event in event_list:
        property_rows.append({"event": event, "property": property['Data attribute'], "type": property['Type'], "required":False, "event_id":None, "property_id":None})
    
df_event_index = pd.DataFrame(property_rows)
df_event_index = df_event_index.sort_values(by=['event'])

# Now search event data for any events with no properties against it (e.g. eol_complete) and add them
event_rows = []
for idx, event in df_events.iterrows():
    if pd.isna(event['Parameters']):
        #print(f"{event['Event name']}: {event['Parameters']}")
        event_rows.append({"event": event['Event name'], "property": None,"type":None, "required":None, "event_id":None, "property_id":None})
df_event_extras = pd.DataFrame(event_rows)
#print(df_event_extras)

df_event_index = pd.concat([df_event_index, df_event_extras], ignore_index=True)
df_event_index = df_event_index.sort_values(by=['event']) 

In [162]:
df_required_properties = pd.DataFrame()

# If file exists, update event_index with correct required values (e.g. True = required)
try:
    df_required_properties = pd.read_csv(required_events_filepath )
except FileNotFoundError: 
    print(f"File '{required_events_filepath}' does not exist, skipping updating event_index")
    df_required_properties = pd.DataFrame()

if not df_required_properties.empty:
    # Update event_index with required properties in events 
    for idx, row in df_required_properties.iterrows():
        # use values row['event'] and row['required_property'] to find matching row in df_event_index, e.g.  df_event_index['event'] and df_event_index['property'] columns
        index_event = df_event_index[(df_event_index['event'] == row['event']) & (df_event_index['property'] == row['required_property'])]
        
        #print(index_event)
        if index_event.empty:
            print(f"Event: '{row['event']}', Property: '{row['required_property']}' not found")
        else:
            df_event_index.loc[index_event.index, 'required'] = True
            

Event: 'abandon_tool', Property: 'fake property' not found
Event: 'fake_event', Property: 'tool' not found


In [163]:
# Create category to tag events with
body = {
    "name": "MAC website",
    "description": "Event stream from MAC website"
}

response = requests.post(f"{RS_API_URL}/v2/catalog/categories", headers=headers, json=body )

if (response.status_code == 200):
    print(f"Category created successfully - id: {response.json()['id']}")
    category_id = response.json()['id']
elif (response.status_code == 400):
    print(f"Category already exists - [{response.status_code}] {response.json()['error']}")
    
    response = requests.get(f"{RS_API_URL}/v2/catalog/categories", headers=headers )
    
    for category in response.json()["data"]:
        if category["name"] == body["name"]:
            category_id = category["id"]
            print("Retrieved category ID: " + category_id)
            break   
else:
    print(f"Error creating category - [{response.status_code}] {response.json()['error']}")
    
    

Category already exists - [400] Category with name MAC website already exists
Retrieved category ID: cat_2yyODDUFmTgMo3sOMSf7RSftHpc


In [ ]:
# Upload properties to data catalog, if property already exists log it and skip to next record

print("Uploading properties to data catalog...")
count = 0
for idx, row in df_properties.iterrows():

    body = {
    "name": row['Data attribute'],
    "description": row['Description'],
    "type": row['Type'],
    }

    response = requests.post(f"{RS_API_URL}/v2/catalog/properties", json=body, headers=headers )

    if response.status_code == 200:
        count += 1
        # Save property id to event index
        if not(save_property_id(row['Data attribute'],row['Type'], response.json()['id'])):
            print(f"Error updating event index for property. Name: {row['Data attribute']}, Type: {row['Type']}, ID: {response.json()['id']})")
    elif response.status_code == 400:
        print(f"Error creating property - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")
        print(f"Searching for existing property...",end="")
        property_id = get_property_id(row['Data attribute'])
        if (property_id):
            print(f"Found existing property '{row['Data attribute']}' with ID: {property_id}")
            # Save property id to event index
            if not(save_property_id(row['Data attribute'],row['Type'], property_id)):
                print(f"Error updating event index for property. Name: {row['Data attribute']}, Type: {row['Type']}, ID: {property_id}")
        else:
            print(f"Error existing property not found - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")
        
    else:
        print(f"Error creating property  - [{response.status_code}] {response.json()['error']} '{row['Data attribute']}'")

print(f"Created {count}/{len(df_properties)} properties")



In [ ]:
# Upload events to data catalog, if event already exists log it and skip to next record
print("Uploading events to data catalog...")

count = 0
body_extras = {}
if category_id != "":
        body_extras["categoryId"] = category_id

for idx, row in df_events.head(10).iterrows():

    # create event
    body = {
    "name": row['Event name'],
    "description": row['Trigger'],
    "eventType": "track",
    **body_extras
    }
   

    response = requests.post(f"{RS_API_URL}/v2/catalog/events", json=body, headers=headers )
    
    if response.status_code == 200:
        count += 1
        # save event id to event index
        if not(save_event_id(row['Event name'], response.json()['id'])):
            print(f"Error updating event index with event. Name: {row['Event name']}, ID: {response.json()['id']}")
    elif response.status_code == 400:
        print(f"Error creating event - [{response.status_code}] {response.json()['error']} '{row['Event name']}'")
        print(f"Searching for existing event...",end="")
        event_id = get_event_id(row['Event name'])
        if (event_id):
            print(f"Found existing event '{row['Event name']}' with ID: {event_id}")
            # Save event id to event index
            if not(save_event_id(row['Event name'], event_id)):
                print(f"Error updating event index for event. Name: {row['Event name']}, ID: {event_id}")
            else:
                print(f"Updated event index for event. Name: {row['Event name']}, ID: {event_id}")
        else:
            print(f"Error existing event not found - [{response.status_code}] {response.json()['error']} '{row['Event name']}'")
    else:
        print(f"Error creating event '{row['Event name']}'- [{response.status_code}] {response.json()['error']}")

print(f"Created {count}/{len(df_events)} events")

In [ ]:
# Create tracking plan
tracking_plan_id = None

body = {
    "name": tracking_plan_name,
    "description": "Tracking plan for MAC website"
}

response = requests.post(f"{RS_API_URL}/v2/catalog/tracking-plans", headers=headers, json=body )
if (response.status_code == 200):
    tracking_plan_id = response.json()["id"]
    print(f"Tracking plan created successfully {tracking_plan_id}")
elif (response.status_code == 400):
    print(f"Failed to create tracking plan {response.json()}")
    print("Searching for existing tracking plan...",end="")
    # See if tracking plan already exists
    response = requests.get(f"{RS_API_URL}/v2/catalog/tracking-plans", headers=headers )
    if response.status_code == 200:
        tracking_plans = response.json()["trackingPlans"]
        if (len(tracking_plans) > 0):
            for tracking_plan in tracking_plans:
                if (tracking_plan["name"] == tracking_plan_name):
                    tracking_plan_id = tracking_plan["id"]
                    print(f"found plan {tracking_plan_id}")
                    break
        else:
            print(f"Error: No plan found ({tracking_plan_name})")
    else:
        print(f"Error: Request failed to get tracking plans {response.status_code}")       
else:
    print(f"Erorr - Failed to create tracking plan {response.status_code}")

print(f"Tracking plan id: {tracking_plan_id}")

In [ ]:
# Populate tracking plan
print("Populating tracking plan...")

# if df_event_index has no 



---
## Utilities

Deletion should be in this order.

Delete the tracking plan to allow event and property deletes






In [216]:
# Delete all properties (not attached to a tracking plan)

get_existing_properties() # Retrieve existing properties from data catalog

count = 0

for idx, row in df_existing_properties.iterrows():
    response = requests.delete(f"{RS_API_URL}/v2/catalog/properties/{row['id']}", headers=headers)
    
    if response.status_code == 200:
        count += 1
    else:
        print(f"Error deleting '{row['name']}' {row['id']}")

print(f"Deleted {count} properties")



Found 2 existing properties
Error deleting 'test1' prop_2yzYpvQntu44Kxyv1cyp7IwYkpA
Deleted 1 properties


In [217]:
# Delete all events (not attached to a tracking plan)

get_existing_events() # get existing events from data catalog

count = 0

for idx, row in df_existing_events.iterrows():
    response = requests.delete(f"{RS_API_URL}/v2/catalog/events/{row['id']}", headers=headers )
    
    if response.status_code == 200:
        count += 1
    else:
        print(f"Error deleting '{row['name']}' {row['id']}")

print(f"Deleted {count} events")

Found 2 existing events
Error deleting 'test_event1' ev_2yzYtmAzHPt5axO75wt2AG5oJzc
Deleted 1 events


In [ ]:
# Remove all events from a tracking plan
# Then, if event isn't on any other tracking plan, delete it

# TODO






Found 10 existing events


In [199]:
df_existing_properties

,propConfig,id,name,description,type,arrayItemTypes,workspaceId,definitionId,itemDefinitionId,createdBy,updatedBy,createdAt,updatedAt
0,{},prop_2yz6GpoNsVXpbchXrSwEmonU4ac,ach_budget_estimate,"For ACH, an object containing the daily fee es...",string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:16:01.370Z,2025-06-25T04:16:01.370Z
1,{},prop_2yz6GfRPsgmSqSiqhM6YhPGBZf7,ach_room_cost,An object containing the outlet name + room ty...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:16:00.174Z,2025-06-25T04:16:00.174Z
2,{},prop_2yz69fZXEZ625t1ldGWRUuRVqqK,active_question,Displays the text associated with the question...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:04.454Z,2025-06-25T04:15:04.454Z
3,{},prop_2yz6Cp0ltOB8uctz1Qa3jkMA7vx,active_screen,The title associated with the current screen b...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:29.985Z,2025-06-25T04:15:29.985Z
4,{},prop_2yz6BIwJEW7lO3UEW1c4Alald0I,age,This parameter shows the age of the person usi...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:17.520Z,2025-06-25T04:15:17.520Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,{},prop_2yz6DDykf8MwjkuPFjf1dQlaq46,user_data,A structured object containing the user data b...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:32.406Z,2025-06-25T04:15:32.406Z
82,{},prop_2yz6A1I5aJDiBt5HcTtldysyCXu,user_type,This parameter tracks weather the user is new ...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:07.378Z,2025-06-25T04:15:07.378Z
83,{},prop_2yz6ENtVczez3SCCJPK3j4hlbOy,uuid,The unique ID associated with the guide (based...,string,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:41.581Z,2025-06-25T04:15:41.581Z
84,{},prop_2yz6Bpg2x3ARdQr84RDy2Oe0YT4,view_count,An integer value that counts how many times a ...,integer,,2xNiB5LbopGv3ktZWbpQXyWEaCg,None,None,2yfWcUocZJfd7iQNO0ct6IlX3WY,None,2025-06-25T04:15:21.512Z,2025-06-25T04:15:21.512Z
